In [ ]:
import codecs
from IPython.display import clear_output
rot_47 = lambda encoded_text: "".join(
    [
        (
            chr(
                (ord(c) - (ord("a") if c.islower() else ord("A")) - 47) % 26
                + (ord("a") if c.islower() else ord("A"))
            )
            if c.isalpha()
            else c
        )
        for c in encoded_text
    ]
)

new_name = rot_47("kmjbmvh_hg")
findme = rot_47(codecs.decode("pbbxa://oqbpcj.kwu/Dqlitvb/qurwg-mtnqvlmz.oqb", "rot_13"))
# uioawhd = rot_47(codecs.decode("pbbxa://oqbpcj.kwu/QIPqaxivw/Ixxtqw.oqb", "rot_13"))
uioawhd = rot_47(codecs.decode("pbbxa://oqbpcj.kwu/Uieqcanx/Ixxtqw.oqb", "rot_13"))
!pip install uv
# !git clone --depth 1 $uioawhd $new_name --branch 3.6.3
!git clone --depth 1 $uioawhd $new_name --branch enhanced
clear_output()
!apt update -y
!apt install -y portaudio19-dev psmisc
!uv pip install -q -r /kaggle/working/program_ml/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system
%cd /kaggle/working/program_ml
!python core.py "prerequisites" --models "True" --exe "True" --pretraineds_hifigan "True" > /dev/null 2>&1
!sudo curl -fsSL https://raw.githubusercontent.com/filebrowser/get/master/get.sh | sudo bash
!filebrowser config init
!filebrowser config set --auth.method=noauth
!filebrowser users add  "applio" "applio123456" --perm.admin
clear_output()
print("Finished")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

model_name = "CarPretrain-ReGAN-wlml-32k-v1"

src = f"/kaggle/input/datasets/juanpelotillas/cprtrgv1/{model_name}"
dst = f"/kaggle/working/program_ml/logs/{model_name}"

os.makedirs(dst, exist_ok=True)

# Files to copy
files = [
    "filelist.txt",
    "config.json",
    "model_info.json",
]

for f in files:
    shutil.copy(os.path.join(src, f), dst)

# Directories to copy
dirs_to_copy = [
    "eval",
]

for d in dirs_to_copy:
    shutil.copytree(
        os.path.join(src, d),
        os.path.join(dst, d),
        dirs_exist_ok=True
    )

# Directories to symlink
dirs_to_link = [
    "f0",
    "f0_voiced",
    "sliced_audios",
    "sliced_audios_16k",
    "extracted",
]

for d in dirs_to_link:
    subprocess.run([
        "ln", "-sfn",
        os.path.join(src, d),
        os.path.join(dst, d)
    ], check=True)

file = Path(dst) / "filelist.txt"
file.write_text(
    file.read_text(encoding="utf-8").replace("\\", "/"),
    encoding="utf-8",
)

print("Done")

In [ ]:
MODEL = "CarPretrain-ReGAN-wlml-32k-v1"  # model_name
SAVE_EVERY = 1                           # save_every_epoch
EPOCHS = 200                             # total_epoch
PRETRAIN_G = "None"                      # pretrained generator
PRETRAIN_D = "None"                      # pretrained discriminator
GPUS = 0                                 # GPU IDs
BATCH = 20                               # batch size
SR = 32000                               # sample rate
SAVE_ONLY_LATEST = False                 # keep only latest checkpoint
SAVE_EVERY_WEIGHTS = False               # save weights every interval
CACHE_GPU = False                        # cache data in VRAM
OVERTRAIN_DETECTOR = True                # enable overtraining detector
OVERTRAIN_THRESHOLD = 50                 # detector threshold
CLEANUP = False                          # delete temporary files
VOCODER = "RefineGAN"                    # vocoder type
CHECKPOINTING = False                    # gradient checkpointing

!cd /kaggle/working/program_ml && python -u /kaggle/working/program_ml/rvc/train/train.py \
{MODEL} {SAVE_EVERY} {EPOCHS} {PRETRAIN_G} {PRETRAIN_D} {GPUS} {BATCH} {SR} \
{SAVE_ONLY_LATEST} {SAVE_EVERY_WEIGHTS} {CACHE_GPU} {OVERTRAIN_DETECTOR} \
{OVERTRAIN_THRESHOLD} {CLEANUP} {VOCODER} {CHECKPOINTING}

In [ ]:
import os
import re
import time
import urllib.request
from IPython.display import clear_output


Tunnel = "Gradio + LocalTunnel" #@param ["Gradio + LocalTunnel", "Ngrok", "LocalTunnel", "Horizon"]
ngrok_authtoken = "" #@param {type:"string"}
horizon_id = "" #@param {type:"string"}


%cd /kaggle/working/program_ml
os.system(f"filebrowser -r /kaggle -p 9876 > /dev/null 2>&1 &")
%load_ext tensorboard
%tensorboard --logdir logs --port 8077

if Tunnel == "Gradio + LocalTunnel":
  print("Using Gradio's built-in tunneling for Applio UI and LocalTunnel for Tensorboard and Filebrowser")
  # gradio  
  share_option = "--share"
  # localtunnel
  # install localtunnel
  !npm install -g localtunnel
  import time
  import urllib
  # run localtunnel
  # tensorboard
  with open('t.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('lt --port 8077 >> t.txt 2>&1 &')

  time.sleep(7)

  endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

  with open('t.txt', 'r') as file:
    t_tunnel = file.read()
    t_tunnel = t_tunnel.replace("your url is: ", "")

  # filebrowser
  with open('f.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('lt --port 9876 >> f.txt 2>&1 &')

  time.sleep(7)

  endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

  with open('f.txt', 'r') as file:
    f_tunnel = file.read()
    f_tunnel = f_tunnel.replace("your url is: ", "")


  clear_output()

  print(f"LocalTunnel Tensorboard Public URL: {t_tunnel}")
  print(f"LocalTunnel Filebrowser Public URL: {f_tunnel}")
  print(f'LocalTunnels Password: {endpoint_ip}')
elif Tunnel == "Ngrok":
  !pip install -q pyngrok
  from pyngrok import ngrok
  ngrok.set_auth_token(ngrok_authtoken)
  p_tunnel = ngrok.connect(6969)
  t_tunnel = ngrok.connect(8077)
  f_tunnel = ngrok.connect(9876)
  clear_output()
  print(f"Applio Public URL: {p_tunnel.public_url}")
  print(f"Tensorboard Public URL: {t_tunnel.public_url}")
  print(f"FileBrowser Public URL: {f_tunnel.public_url}")
elif Tunnel == "LocalTunnel":
  # install
  !npm install -g localtunnel
  import time
  import urllib
  # run localtunnel
  # program_ml
  with open('p.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('lt --port 6969 >> p.txt 2>&1 &')

  time.sleep(7)

  endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

  with open('p.txt', 'r') as file:
    p_tunnel = file.read()
    p_tunnel = p_tunnel.replace("your url is: ", "")

  # tensorboard
  with open('t.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('lt --port 8077 >> t.txt 2>&1 &')

  time.sleep(7)

  endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

  with open('t.txt', 'r') as file:
    t_tunnel = file.read()
    t_tunnel = t_tunnel.replace("your url is: ", "")

  # filebrowser
  with open('f.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('lt --port 9876 >> f.txt 2>&1 &')

  time.sleep(7)

  endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

  with open('f.txt', 'r') as file:
    f_tunnel = file.read()
    f_tunnel = f_tunnel.replace("your url is: ", "")


  clear_output()

  print(f"Applio Public URL: {p_tunnel}")
  print(f"Tensorboard Public URL: {t_tunnel}")
  print(f"Filebrowser Public URL: {f_tunnel}")
  print(f'LocalTunnels Password: {endpoint_ip}')
elif Tunnel == "Horizon":
  # install 
  !npm install -g @hrzn/cli
  import time
  # login
  !hrzn login $horizon_id
  # run horizon
  # program_ml
  with open('p.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('hrzn tunnel http://localhost:6969 >> p.txt 2>&1 &')

  time.sleep(7)

  with open('p.txt', 'r') as file:
    p_tunnel = file.read()
    p_tunnel = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" p.txt
    p_tunnel = p_tunnel[0]

  # tensorboard
  with open('t.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('hrzn tunnel http://localhost:8077 >> t.txt 2>&1 &')

  time.sleep(7)

  with open('t.txt', 'r') as file:
    t_tunnel = file.read()
    t_tunnel = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" t.txt
    t_tunnel = t_tunnel[0]

  # filebrowser
  with open('f.txt', 'w') as file:
    file.write('')

  get_ipython().system_raw('hrzn tunnel http://localhost:9876 >> f.txt 2>&1 &')

  time.sleep(7)

  with open('f.txt', 'r') as file:
    f_tunnel = file.read()
    f_tunnel = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" f.txt
    f_tunnel = f_tunnel[0]

  clear_output()

  print(f"Applio Public URL: {p_tunnel}")
  print(f"Tensorboard Public URL: {t_tunnel}")
  print(f"FileBrowser Public URL: {f_tunnel}")

print("Save your links for later, this will take a while...")
!python app.py --client --host 0.0.0.0 --port 6969 $share_option

# kills previously running processes
!fuser -k 6969/tcp
!fuser -k 8077/tcp
!fuser -k 9876/tcp

In [ ]:
"""
!mkdir -p /kaggle/working/program_ml/logs/modelname && \
cd /kaggle/working/program_ml/logs/CarPretrain-ReGAN-wlml-32k-v1 && \
cp /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/filelist.txt . && \
cp /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/config.json . && \
cp /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/model_info.json . && \
cp -r /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/eval . && \
ln -sfn /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/f0 f0 && \
ln -sfn /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/f0_voiced f0_voiced && \
ln -sfn /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/sliced_audios sliced_audios && \
ln -sfn /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/sliced_audios_16k sliced_audios_16k && \
ln -sfn /kaggle/input/datasets/juanpelotillas/cprtrgv1/CarPretrain-ReGAN-wlml-32k-v1/extracted extracted
"""

In [ ]:
model_name = "CarPretrain-ReGAN-wlml-32k-v1"
files = [
    "D_5698.pth", "G_5698.pth"
]

for f in files:
    !cd "/kaggle/working/program_ml/logs/{model_name}" && curl -O "xxxxxxxxxx:39182/{f}"

print(f"Done")